**Notebook zur Analyse und Behebung eines kritischen Regression-Bugs in der Rezept-Skalierung einer Website.**

Dieses Notebook dokumentiert die Schritte zur Fehlersuche und Behebung.

## 1) Lokale Entwicklungsumgebung starten

- Starte die Website lokal (z. B. `Live Server` oder einfacher: öffne `index.html` im Browser).
- Falls Node/Dev-Server erforderlich: starte `npm start` oder `npx http-server`.
- Prüfe Terminal-Ausgaben auf Fehler beim Laden oder fehlende Abhängigkeiten.

```python
# Beispiel: simple HTTP Server (falls nötig)
# python3 -m http.server 8000
```

## 2) Browser-Konsole und Terminal-Fehler prüfen

- Öffne die Browser-Konsole (F12) und lade die Seite neu.
- Notiere die erste auftretende Fehlermeldung (Stacktrace, Datei, Zeilennummer).
- Prüfe Terminal-Logs (Server) auf Ladefehler.

**Typische Fehler**: `Uncaught TypeError: Cannot read property 'map' of undefined`, `ReferenceError: detailOriginalNutrition is not defined`, SyntaxError, etc.

## 3) Letzte Änderungen an Skalierungskomponenten untersuchen

- Öffne `script.js` und suche nach den zuletzt eingefügten Funktionen: `formatScaleInputValue`, `detailOriginalNutrition`, `setRecipeDetailScale`, `scale-factor-display`, `portion-control`.
- Prüfe, ob diese Funktionen globale Seiteneffekte in anderen Teilen verursachen.
- Suche nach Stellen, an denen `recipe.ingredients` ohne Fallback genutzt wird.

## 4) Regressionsursache in Rezeptübersicht identifizieren

- Prüfe `displayRecipes()` und `showRecipeDetail()` auf Annahmen über `recipe`-Struktur.
- Achte auf Funktionsaufrufe, die beim Rendern der Übersicht fehlschlagen könnten, z. B. `normalizeIngredientLines(recipe)` oder `detailOriginalNutrition = {...}` die außerhalb von `showRecipeDetail` referenziert werden.
- Suche nach unbedachten Variablen, die global sind und beim Laden der Übersicht `undefined` sein könnten.

## 5) Minimalen Fix für Rezept-Skalierung anwenden

- Füge defensive Checks hinzu: z. B. `const ingredients = Array.isArray(recipe.ingredients) ? recipe.ingredients : (recipe.ingredients || '').split(',')`.
- Stelle sicher, dass `detailOriginalNutrition` nur in `showRecipeDetail` initialisiert wird und nicht beim Laden der Übersicht benutzt wird.
- Schütze `formatScaledQuantity` und `setRecipeDetailScale` vor `undefined` Daten.

## 6) Funktionalität von Rezeptkarten und Filtermenü prüfen

- Lade die Startseite und überprüfe, ob `displayRecipes(filteredRecipes)` wieder erfolgreich `recipe-card` Elemente erstellt.
- Teste Filter öffnen und schließen (`#filter-overlay` und `#filter-popup`) und stelle sicher, dass Event-Handler intakt sind.

## 7) Rezeptdetailseite und Skalierung verifizieren

- Öffne ein Rezept, teste das Portionen-Control und beobachte die Konsole.
- Überprüfe, dass fehlende `calories` oder `protein` keine Exceptions erzeugen.
- Stelle sicher, dass Skalierung nur in `showRecipeDetail` wirkt und nicht in `displayRecipes`.